# Fase 3: Dataset completo, entrenamiento y validación end-to-end

Corre el pipeline completo (presegmentación → cuaterniones → QFT → características → etiquetado) sobre varias imágenes, junta todo en `X_total`/`y_total`, normaliza, separa train/test y entrena el primer clasificador.

### Setup

In [1]:
import sys; sys.path.append('..')

from src.io_utils import cargar_imagen, cargar_mascara, listar_pares_imagen_mascara
from src.presegmentation import presegmentar_superpixeles
from src.quaternion_qft import procesar_qft_superpixeles
from src.feature_extraction import extraer_matriz_caracteristicas, normalizar_caracteristicas
from src.classification import etiquetar_superpixeles, entrenar_modelo, predecir

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

import numpy as np
import matplotlib.pyplot as plt

pares = listar_pares_imagen_mascara('../data/raw/train', '../data/masks/train')
print("Total de pares imagen/máscara disponibles:", len(pares))

Total de pares imagen/máscara disponibles: 900


## Subconjunto de trabajo

Empieza con pocas imágenes para validar que el flujo completo corre sin errores y en tiempo razonable, antes de lanzarlo sobre todo el dataset. Sube `N_IMAGENES` (o ponlo en `len(pares)`) cuando ya confirmes que funciona.

In [ ]:
N_IMAGENES = 20  # cámbialo a len(pares) para correr sobre todo el dataset
N_SEGMENTOS = 100
UMBRAL_ETIQUETADO = 0.5

pares_subset = pares[:N_IMAGENES]
print(f"Procesando {len(pares_subset)} imágenes...")

## Loop principal: pipeline completo por imagen

Acumula `X` y `y` de cada imagen. Si alguna imagen falla (SLIC/QFT), se reporta pero no detiene el loop.

In [ ]:
X_total = []
y_total = []
imagenes_fallidas = []

for i, (ruta_img, ruta_mask) in enumerate(pares_subset):
    try:
        img = cargar_imagen(ruta_img)
        mask = cargar_mascara(ruta_mask)

        regiones = presegmentar_superpixeles(img, n_segmentos=N_SEGMENTOS)
        espectros = procesar_qft_superpixeles(img, regiones)

        X_img = extraer_matriz_caracteristicas(espectros)
        y_img = etiquetar_superpixeles(regiones, mask, umbral=UMBRAL_ETIQUETADO)

        X_total.append(X_img)
        y_total.append(y_img)

    except Exception as e:
        imagenes_fallidas.append((ruta_img, str(e)))

    if (i + 1) % 5 == 0 or (i + 1) == len(pares_subset):
        print(f"  {i + 1}/{len(pares_subset)} imágenes procesadas")

if imagenes_fallidas:
    print(f"\n⚠️  {len(imagenes_fallidas)} imágenes fallaron:")
    for ruta, err in imagenes_fallidas:
        print(f"   {ruta}: {err}")

X_total = np.vstack(X_total)
y_total = np.concatenate(y_total)

print("\nX_total:", X_total.shape)
print("y_total:", y_total.shape, "| clase 0:", np.sum(y_total == 0), "| clase 1:", np.sum(y_total == 1))

### Revisar balance de clases

Si una clase domina fuertemente (ej. 90/10), conviene tenerlo en cuenta al entrenar (class_weight='balanced' en el modelo, o ajustar el umbral de etiquetado).

In [ ]:
plt.bar(['Piel sana (0)', 'Lesión (1)'], [np.sum(y_total == 0), np.sum(y_total == 1)])
plt.title('Balance de clases en el conjunto acumulado')
plt.ylabel('Número de superpíxeles')
plt.show()

print(f"Proporción clase 1: {np.mean(y_total == 1):.1%}")

## Normalización

Se normaliza **después** de juntar todas las imágenes, no por imagen — para que las características sean comparables entre sí en la misma escala.

In [ ]:
X_norm_total = normalizar_caracteristicas(X_total, metodo='zscore')
X_norm_total[:5]

## Split train/test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_norm_total, y_total, test_size=0.2, random_state=42, stratify=y_total
)

print("Train:", X_train.shape, "| Test:", X_test.shape)

## Entrenar y comparar los 3 modelos

In [ ]:
resultados = {}

for tipo in ["svm", "knn", "random_forest"]:
    modelo = entrenar_modelo(X_train, y_train, tipo_modelo=tipo)
    y_pred = predecir(modelo, X_test)
    resultados[tipo] = (modelo, y_pred)

    print(f"\n=== {tipo.upper()} ===")
    print(classification_report(y_test, y_pred, target_names=['piel sana', 'lesión']))

### Matrices de confusión

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (tipo, (modelo, y_pred)) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['piel sana', 'lesión'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(tipo.upper())

plt.tight_layout()
plt.show()

## Siguiente paso

Con el mejor modelo de `resultados` (por f1-score o el criterio que definan) ya se puede pasar a **reconstrucción de máscara** (repintar la imagen según las predicciones por superpíxel) y **evaluación** (IoU, Dice, precisión, recall) contra el ground-truth real.